In [1]:
from functools import lru_cache

#### Terminology

| Term | What it is | Example for ಕ |
| :--- | ---: | ---: |
| Code point | 	The number Unicode assigns to a character | ord('ಕ') = 3221 (U+0C95) |
| Encoding (UTF-8) | The rule that turns a code point into bytes | 'ಕ'.encode('utf-8') = b'\xe0\xb2\x95' |
| Byte | A raw value from 0 to 255 | [224, 178, 149] |
| Byte-level representation or <br> byte-level characters or <br> byte-level symbols | GPT-2’s trick: each byte shown as one printable character (display character) | 'à²ķ' |

In [2]:
print(ord('ಕ'))
print(ord('G'))

print(f"\n{'ಕ'.encode('utf-8')}")
print('k'.encode('utf-8'))


3221
71

b'\xe0\xb2\x95'
b'k'


In [ ]:
@lru_cache()
def bytes_to_unicode():
    """
    This is the gpt2-tokenizer's function.
    It changes the mapping of bytes to characters (from what unicode normally uses).
    In the first 256 characters, unicode contains 188 printable bytes and 68 whitespace/invisible characters.
    This function moves the invisible/whitespace characters to 256-323 so they become visible.
    Eg: Byte 32 (space) is displayed as Ġ (288), byte 10 (newline) is displayed as Ċ (266).

    Returns list of utf-8 byte and a corresponding list of unicode strings.
    The reversible bpe codes work on unicode strings.
    This means you need a large # of unicode characters in your vocab if you want to avoid UNKs.
    When you're at something like a 10B token dataset you end up needing around 5K for decent coverage.
    This is a signficant percentage of your normal, say, 32K bpe vocab.
    To avoid that, we want lookup tables between utf-8 bytes and unicode strings.
    And avoids mapping to whitespace/control characters the bpe code barfs on.
    """
    bs = list(range(ord("!"), ord("~")+1))+list(range(ord("¡"), ord("¬")+1))+list(range(ord("®"), ord("ÿ")+1))
    cs = bs[:]
    n = 0
    for b in range(2**8):
        if b not in bs:
            bs.append(b)
            cs.append(2**8+n)
            n += 1
            #print(f"This b is not in bs: {b}; it's char is: '{chr(b)}'")
    cs = [chr(n) for n in cs]
    return dict(zip(bs, cs))


The bytes_to_unicode function is usually called the byte encoder. It provides the display character for a byte.

In [4]:
# This inverts the byte encoder: bytes_to_unicode() maps byte (0-255) -> display char
# so this maps display char -> original byte.
byte_decoder = {v: k for k, v in bytes_to_unicode().items()}


def to_bytes(s):
    # In the line of code below, 's' is a byte-level token string like 'Ġà²ķ'.
    # Each character stands for one raw byte. Look each one up to recover the original bytes (by using byte_decoder)
    return bytes(byte_decoder[c] for c in s)


def to_text(s):
    # Interpret the recovered bytes as UTF-8 to get real text back.
    # errors="replace" inserts '�' if the token ends mid-character, which is common for Kannada, where every letter is 3 bytes.
    return to_bytes(s).decode("utf-8", errors="replace")


In [5]:
print(byte_decoder["à"])            # Note: 'à' is the first byte of every 3-byte utf-8 character in the U+0800–U+0FFF range, which includes all of Kannada.
                                    # Kannada characters are in the U+0C80 to U+0CFF range; it is the logical block assigned to Kannada in unicode
print(bytes(byte_decoder["à"]))

print(f"\n{byte_decoder["Ġ"]}")
print(bytes(byte_decoder["Ġ"]))

# Ġà means a space, followed by the first third of some Kannada letter. It doesn't represent a full kannada character yet.

224
b'\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00'

32
b'\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x

In [6]:
print(to_text("Ġà²Ī"))
# Checking if 'Ġ' is a space (it is)
print(to_text("à²Ī"))

print(to_text("Ġà²®à²¤à³įà²¤à³ģ"))
# Checking if 'à' indicates the start of a kannada character
print(to_text("à²®"), " + ", to_text("à²¤"), " + ", to_text("à³į"), " + ", to_text("à²¤"), " + ", to_text("à³ģ"))
# Note that 'à²' indicates the start of a regular character, whereas 'à³' indicates the start of an ottakshara
# Also note that it takes three bytes to form a full character; if it's only two characters, that doesn't form a full word
print(to_text("à²"))

print(to_text("Ġà²İà²Ĥà²¦à³ģ"))
print(to_text("Ġà²ħà²µà²°à³ģ"))
print(to_text("Ġà²¹à²¾à²Ĺà³Ĥ"))

 ಈ
ಈ
 ಮತ್ತು
ಮ  +  ತ  +  ್  +  ತ  +  ು
�
 ಎಂದು
 ಅವರು
 ಹಾಗೂ


In [8]:
len(bytes_to_unicode().keys())      # This returns 256 characters, as expected

256

In [9]:
bytes_to_unicode()

{33: '!',
 34: '"',
 35: '#',
 36: '$',
 37: '%',
 38: '&',
 39: "'",
 40: '(',
 41: ')',
 42: '*',
 43: '+',
 44: ',',
 45: '-',
 46: '.',
 47: '/',
 48: '0',
 49: '1',
 50: '2',
 51: '3',
 52: '4',
 53: '5',
 54: '6',
 55: '7',
 56: '8',
 57: '9',
 58: ':',
 59: ';',
 60: '<',
 61: '=',
 62: '>',
 63: '?',
 64: '@',
 65: 'A',
 66: 'B',
 67: 'C',
 68: 'D',
 69: 'E',
 70: 'F',
 71: 'G',
 72: 'H',
 73: 'I',
 74: 'J',
 75: 'K',
 76: 'L',
 77: 'M',
 78: 'N',
 79: 'O',
 80: 'P',
 81: 'Q',
 82: 'R',
 83: 'S',
 84: 'T',
 85: 'U',
 86: 'V',
 87: 'W',
 88: 'X',
 89: 'Y',
 90: 'Z',
 91: '[',
 92: '\\',
 93: ']',
 94: '^',
 95: '_',
 96: '`',
 97: 'a',
 98: 'b',
 99: 'c',
 100: 'd',
 101: 'e',
 102: 'f',
 103: 'g',
 104: 'h',
 105: 'i',
 106: 'j',
 107: 'k',
 108: 'l',
 109: 'm',
 110: 'n',
 111: 'o',
 112: 'p',
 113: 'q',
 114: 'r',
 115: 's',
 116: 't',
 117: 'u',
 118: 'v',
 119: 'w',
 120: 'x',
 121: 'y',
 122: 'z',
 123: '{',
 124: '|',
 125: '}',
 126: '~',
 161: '¡',
 162: '¢',
 163: '£',

#### Why the above functions are able to return Kannada characters

"à²ķ" --byte_decoder-->  [224, 178, 149]  --UTF-8 decode-->  "ಕ"

Remember that the characters "à²ķ" are already following the mapping represented by the bytes_to_unicode function.

For character Ġ, the code point is 288, whereas the byte is 32 (shown below). The byte_decoder dictionary correctly returns 32. <br>This is interpreted as a space by the UTF-8 decoder.<br><br>
For à, both the code point and byte are 224. The UTF-8 decoder interprets that as the first byte of ಕ.

In [10]:
print(byte_decoder['Ġ'])
print(ord('Ġ'))

32
288


### encode, decode, bytes()

* `.encode('utf-8')` runs on a str. It turns each character into its UTF-8 byte sequence.
* `.decode('utf-8')` runs on bytes. It reads the bytes according to UTF-8 rules and builds characters.
* `bytes(...)` is the constructor for the bytes type. Give it a list of integers 0–255 and it packs them into a bytes object

In [11]:
b = bytes([224, 178, 149])

print(b)                  # b'\xe0\xb2\x95'
print(list(b))            # [224, 178, 149]   <- it really is just integers
print(b.decode('utf-8'))  # ಕ

b'\xe0\xb2\x95'
[224, 178, 149]
ಕ


In [12]:
print("a".encode('utf-8'))
# The output, b'a' is the single byte 97. Python displays a byte as its ASCII character when it has one, and as a \x.. hex when it doesn't.


# Confirming that the byte value of a is 97:
print(list(b'a'))

b'a'
[97]


In to_text, to_bytes returns a bytes object, and .decode turns it into Kannada.

In [13]:
print('a'.encode('utf-8'))     # b'a'
print('à²ķ'.encode('utf-8'))   # b'\xc3\xa0\xc2\xb2\xc4\xb7'

b'a'
b'\xc3\xa0\xc2\xb2\xc4\xb7'


Calling .encode on the display string treats à, ², ķ as real Latin characters and UTF-8-encodes each one:
- 'à' (U+00E0) → C3 A0
- '²' (U+00B2) → C2 B2
- 'ķ' (U+0137) → C4 B7

The result is 6 bytes (two bytes per character) that have nothing to do with ಕ. If you decode them, you get 'à²ķ' back, not Kannada.

That’s why to_bytes uses the byte_decoder table instead of .encode(). The two take different paths and give different results:
- 'à²ķ' → byte_decoder lookup → [224, 178, 149]        → decode → 'ಕ'   ✓
- 'à²ķ' → .encode('utf-8')    → [195,160,194,178,196,183] → decode → 'à²ķ' ✗

### Why we use GPT-2's bytes_to_unicode() when the model is OLMo

OLMo’s tokenizer, like most modern Hugging Face BPE tokenizers (GPT-2, RoBERTa, Llama 3, Qwen), includes a component called the ByteLevel pre-tokenizer. ByteLevel uses the exact bytes_to_unicode table from GPT-2. The vocabularies differ between these models, but the byte-to-character table is shared.

Confirming this by pretokenizing the same string with both:

In [14]:
from transformers import AutoTokenizer

model_dir = f"/Users/nirajh/Code/Language_adaptation/olmo2-7b-stage1"
tok = AutoTokenizer.from_pretrained(model_dir)

/opt/miniconda3/envs/py312/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [15]:
print(tok.backend_tokenizer.pre_tokenizer.pre_tokenize_str(" ಕ")[0][0])

Ġà²ķ


In [16]:
encoded_bytes = ' ಕ'.encode('utf-8')
print(''.join(bytes_to_unicode()[b] for b in list(encoded_bytes)))


Ġà²ķ
